## Assignment 2: Probabilistic Models and Vector Space Applications

### Assignment Overview

This assignment builds on fundamental text processing techniques to explore probabilistic language modeling, text classification, and the practical application of vector space models for information retrieval. You will implement a simple n-gram language model, build a complete text classification pipeline, develop a search engine, and analyze the core components of sequence models.

You are required to complete five coding-related tasks. For each task, you will be working with a specified dataset or corpus. Please submit your solutions in this single Jupyter Notebook (`.ipynb`) file, clearly marking each task. Ensure your code is well-commented and your findings are explained in markdown cells where requested.

### Task 1: Implementing a Bigram Language Model with Laplace Smoothing (20 Marks)

**Objective:** To understand the fundamentals of n-gram language models, including probability calculation, smoothing, and evaluation with perplexity.

**Description:** You will implement a Bigram language model from scratch. Your model will be trained on a small corpus and will use Add-One (Laplace) smoothing to handle unseen n-grams.

**Your task is to:**

1.  **Implement a training function `train_bigram_model(corpus)`:**
    * The corpus will be a list of sentences, where each sentence is a list of tokens.
    * The function should count all unigrams and bigrams in the corpus.
    * It should return the unigram counts, bigram counts, and the vocabulary size (V).

2.  **Implement a probability function `calculate_bigram_prob(prev_word, word, unigram_counts, bigram_counts, V)`:**
    * This function should calculate the smoothed probability of a `word` given the `prev_word` using the formula for Laplace (Add-One) smoothing: $P(w_i | w_{i-1}) = \frac{C(w_{i-1}, w_i) + 1}{C(w_{i-1}) + V}$.

3.  **Implement a perplexity calculation function `calculate_perplexity(sentence, ...)`:**
    * This function should take a test sentence and your trained model components as input.
    * It should calculate the perplexity of the sentence using the formula: $PP(W) = P(w_1, w_2, ..., w_N)^{-1/N}$. Remember to handle the start of the sentence appropriately (e.g., by assuming a start token `<S>`).

4.  **Train and Evaluate:**
    * Train your model on the provided `train_corpus`.
    * Calculate and print the perplexity of your model on the `test_sentence`.

**Corpus:**

```python
# Sample corpus for training and testing
train_corpus = [["<S>", "i", "am", "sam", "</S>"], ["<S>", "sam", "i", "am", "</S>"], ["<S>", "i", "do", "not", "like", "green", "eggs", "and", "ham", "</S>"]]
test_sentence = ["<S>", "i", "like", "green", "ham", "</S>"]
```

In [1]:
import numpy as np
from collections import Counter, defaultdict

train_corpus = [["<S>", "i", "am", "sam", "</S>"], 
                ["<S>", "sam", "i", "am", "</S>"], 
                ["<S>", "i", "do", "not", "like", "green", "eggs", "and", "ham", "</S>"]]
test_sentence = ["<S>", "i", "like", "green", "ham", "</S>"]

def train_bigram_model(corpus):
    unigram_counts = Counter()
    bigram_counts = Counter()
    
    for sentence in corpus:
        unigram_counts.update(sentence)
        for i in range(len(sentence) - 1):
            bigram = (sentence[i], sentence[i+1])
            bigram_counts[bigram] += 1
            
    V = len(unigram_counts)
    return unigram_counts, bigram_counts, V

def calculate_bigram_prob(prev_word, word, unigram_counts, bigram_counts, V):
    count_bigram = bigram_counts.get((prev_word, word), 0)
    count_unigram = unigram_counts.get(prev_word, 0)
    prob = (count_bigram + 1) / (count_unigram + V)
    return prob

def calculate_perplexity(sentence, unigram_counts, bigram_counts, V):
    log_prob_sum = 0.0
    N = len(sentence) - 1  # exclude <S> as start context
    for i in range(1, len(sentence)):
        p = calculate_bigram_prob(sentence[i-1], sentence[i], unigram_counts, bigram_counts, V)
        if p == 0:
            return float('inf')
        log_prob_sum += np.log(p)
    perplexity = np.exp(-log_prob_sum / N)
    return perplexity

unigram_counts, bigram_counts, V = train_bigram_model(train_corpus)
perplexity = calculate_perplexity(test_sentence, unigram_counts, bigram_counts, V)
print(f"Perplexity of the test sentence: {perplexity:.2f}")

Perplexity of the test sentence: 8.37


### Task 2: Text Classification with TF-IDF and Naive Bayes (20 Marks)

**Objective:** To build a complete text classification pipeline using TF-IDF feature extraction and a Multinomial Naive Bayes classifier.

**Description:** You will use `scikit-learn` to classify SMS messages as either "spam" or "ham" (not spam). This task integrates vector space representation with a classic probabilistic model.

**Your task is to:**

1.  Load the SMS Spam Collection dataset.
2.  Split the dataset into an 80% training set and a 20% testing set.
3.  Create a text processing pipeline using `sklearn.pipeline.Pipeline` that consists of two steps:
    * `TfidfVectorizer`: To convert text messages into TF-IDF vectors. Use the default parameters.
    * `MultinomialNB`: The Multinomial Naive Bayes classifier.
4.  Train the pipeline on the training data.
5.  Evaluate the trained model on the testing data. Print the following:
    * The accuracy of the model.
    * A full classification report (including precision, recall, and F1-score for each class) using `sklearn.metrics.classification_report`.
6.  Use the trained pipeline to predict the class of two new messages: `"Congratulations! You've won a $1,000 gift card. Go to http://example.com to claim now."` and `"Hi mom, I'll be home for dinner tonight."`

**Dataset:**

* **SMS Spam Collection Dataset:** A public set of SMS labeled messages.
* **Access:** Download from the UCI Machine Learning Repository: [SMS Spam Collection](https://archive.ics.uci.edu/ml/datasets/sms+spam+collection). You will need the `SMSSpamCollection` file.

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

# Load the dataset directly from the downloaded file
df = pd.read_csv('SMSSpamCollection', sep='\t', header=None, names=['label', 'message'])

# Rest of the code remains unchanged
X_train, X_test, y_train, y_test = train_test_split(
    df['message'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('nb', MultinomialNB())
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

new_messages = [
    "Congratulations! You've won a $1,000 gift card. Go to http://example.com to claim now.",
    "Hi mom, I'll be home for dinner tonight."
]
predictions = pipeline.predict(new_messages)
for msg, pred in zip(new_messages, predictions):
    print(f"Message: {msg[:50]}... -> Prediction: {pred}")

Accuracy: 0.9605381165919282

Classification Report:
               precision    recall  f1-score   support

         ham       0.96      1.00      0.98       966
        spam       1.00      0.70      0.83       149

    accuracy                           0.96      1115
   macro avg       0.98      0.85      0.90      1115
weighted avg       0.96      0.96      0.96      1115

Message: Congratulations! You've won a $1,000 gift card. Go... -> Prediction: spam
Message: Hi mom, I'll be home for dinner tonight.... -> Prediction: ham


### Task 3: Building a Simple Information Retrieval System (20 Marks)

**Objective:** To apply TF-IDF and Cosine Similarity to build a basic document retrieval system that ranks documents based on their relevance to a query.

**Description:** You will create a system that takes a text query and returns the most relevant documents from a small corpus. This is the core principle behind search engines.

**Your task is to:**

1.  Use the provided `document_corpus`.
2.  Create a `TfidfVectorizer` and fit it on the corpus to learn the vocabulary and IDF weights.
3.  Transform the corpus into a TF-IDF document-term matrix.
4.  Write a function `rank_documents(query, vectorizer, doc_term_matrix, top_n=3)` that:
    * Takes a `query` string, the fitted `vectorizer`, the document-term `matrix`, and an optional `top_n` integer.
    * Transforms the input query into a TF-IDF vector using the *same* vectorizer.
    * Calculates the cosine similarity between the query vector and all document vectors in the matrix.
    * Returns the indices and content of the `top_n` most similar documents.
5.  Demonstrate your system by running it with the query `"deep learning models for vision"` and printing the ranked results.

**Dataset:**

```python
# A small corpus of document abstracts
document_corpus = [
    "The field of machine learning has seen rapid growth in recent years, especially in deep learning.",
    "Natural language processing allows machines to understand and respond to human text.",
    "Computer vision focuses on enabling computers to see and interpret the visual world.",
    "Deep learning models like convolutional neural networks are powerful for computer vision tasks.",
    "Recurrent neural networks are often used for sequential data in natural language processing."
    ...
]
```

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

document_corpus = [
    "The field of machine learning has seen rapid growth in recent years, especially in deep learning.",
    "Natural language processing allows machines to understand and respond to human text.",
    "Computer vision focuses on enabling computers to see and interpret the visual world.",
    "Deep learning models like convolutional neural networks are powerful for computer vision tasks.",
    "Recurrent neural networks are often used for sequential data in natural language processing.",
    "The advances in reinforcement learning have led to breakthroughs in game playing and robotics.",
    "Transfer learning enables models trained on large datasets to be adapted for new tasks with limited data.",
    "Unsupervised learning techniques can discover hidden patterns in data without labeled examples.",
    "Optimization algorithms such as stochastic gradient descent are crucial for training neural networks.",
    "Attention mechanisms have improved the performance of natural language translation and image captioning.",
    "Generative adversarial networks create realistic images and are used for data augmentation.",
    "Feature engineering and selection are important steps in classical machine learning pipelines.",
    "Object detection is a key task in computer vision that involves locating instances within images.",
    "The combination of convolutional and recurrent networks is used for video classification tasks.",
    "Zero-shot learning allows models to recognize objects and concepts they have not seen during training.",
    "Natural language generation is used for creating text summaries and chatbot responses.",
    "Graph neural networks leverage graph structures for tasks such as social network analysis and chemistry.",
    "Hyperparameter tuning can significantly improve the accuracy of deep learning models.",
    "Cross-modal learning involves integrating information from multiple data sources such as text and images.",
    "Evaluating model performance requires a good choice of metrics such as F1-score and RMSE."
]

vectorizer = TfidfVectorizer()
doc_term_matrix = vectorizer.fit_transform(document_corpus)

def rank_documents(query, vectorizer, doc_term_matrix, top_n=3):
    query_vec = vectorizer.transform([query])
    similarities = cosine_similarity(query_vec, doc_term_matrix).flatten()
    top_indices = np.argsort(similarities)[::-1][:top_n]
    return [(idx, document_corpus[idx]) for idx in top_indices]

# 测试查询
query = "deep learning models for vision"
ranked_docs = rank_documents(query, vectorizer, doc_term_matrix, top_n=3)

print(f"Top {len(ranked_docs)} documents for the query: '{query}'\n")
for i, (idx, doc) in enumerate(ranked_docs):
    print(f"Rank {i+1}: {doc}")

Top 3 documents for the query: 'deep learning models for vision'

Rank 1: Deep learning models like convolutional neural networks are powerful for computer vision tasks.
Rank 2: Hyperparameter tuning can significantly improve the accuracy of deep learning models.
Rank 3: The field of machine learning has seen rapid growth in recent years, especially in deep learning.


### Task 4: Implementing Viterbi for HMM POS Tagging (20 Marks)

**Objective:** Implement a simple Hidden Markov Model (HMM) POS tagger via the Viterbi algorithm.

**Description:** Implement Viterbi decoding for a small HMM and apply it to two sentences with the ambiguous word "book". Then briefly discuss why HMMs work for POS tagging and a limitation of the Markov assumption.

**Your task is to:**

1. Define two sentences:
    * `sentence1 = "The book is on the table."`
    * `sentence2 = "I want to book a flight."`
2. Implement Viterbi in log-space for a small tag set (e.g., `{DET, NOUN, VERB, PRT}`). Use the example initial (π), transition (A), and emission (B) probabilities in the parameters block below, or define your own consistent matrices and document them.
3. Run your decoder on both sentences and print the predicted tag sequence and total log-probability.
4. In a markdown cell, explain:
    * **a)** How transition and emission probabilities lead to different tags for "book" in the two sentences.
    * **b)** One sentence where the first-order Markov assumption is limiting, and why.

**Parameters:** Use the matrices shown in the section “Viterbi decoding for a simple HMM (Task 4)” below.

#### Viterbi decoding for a simple HMM (Task 4)

We illustrate HMM POS tagging with a small tag set `T = {DET, NOUN, VERB, PRT}` and vocabulary `V = {the, a, book, table, flight, is, want, to, on, i}`. The HMM comprises initial probabilities π, tag-to-tag transitions A, and tag-to-word emissions B.

Example parameters (each row sums to 1):

- Initial π:
  - P(DET)=0.50, P(NOUN)=0.20, P(VERB)=0.20, P(PRT)=0.10

- Transition A (rows: from-tag, cols: to-tag) in order [DET, NOUN, VERB, PRT]:

```text
from\to   DET    NOUN   VERB   PRT
DET      0.05   0.75   0.15   0.05
NOUN     0.05   0.10   0.75   0.10
VERB     0.10   0.35   0.40   0.15
PRT      0.05   0.10   0.75   0.10
```

- Emission B:
  - DET: the(0.80), a(0.20)
  - NOUN: book(0.45), table(0.25), flight(0.20), i(0.05), on(0.05)
  - VERB: is(0.40), want(0.35), book(0.20), to(0.03), on(0.02)
  - PRT: to(0.70), on(0.30)

Viterbi recurrence in log-space to avoid underflow:

- Initialization: `V[tag, 0] = log π[tag] + log B[tag, x0]`
- Recurrence: `V[tag, i] = log B[tag, xi] + max_prev ( V[prev, i-1] + log A[prev->tag] )`
- Backtrace from the best final tag.

We will decode the most likely tag sequence for the two Task 4 sentences using these parameters.


In [6]:
import math
from typing import List, Tuple

TAGS = ["DET", "NOUN", "VERB", "PRT"]

# 初始概率 π
pi = {
    "DET": 0.50,
    "NOUN": 0.20,
    "VERB": 0.20,
    "PRT": 0.10
}

# 转移概率 A
A = {
    "DET": {"DET": 0.05, "NOUN": 0.75, "VERB": 0.15, "PRT": 0.05},
    "NOUN": {"DET": 0.05, "NOUN": 0.10, "VERB": 0.75, "PRT": 0.10},
    "VERB": {"DET": 0.10, "NOUN": 0.35, "VERB": 0.40, "PRT": 0.15},
    "PRT": {"DET": 0.05, "NOUN": 0.10, "VERB": 0.75, "PRT": 0.10}
}

# 发射概率 B
B = {
    "DET": {"the": 0.80, "a": 0.20},
    "NOUN": {"book": 0.45, "table": 0.25, "flight": 0.20, "i": 0.05, "on": 0.05},
    "VERB": {"is": 0.40, "want": 0.35, "book": 0.20, "to": 0.03, "on": 0.02},
    "PRT": {"to": 0.70, "on": 0.30}
}

UNK = 1e-8

def emission_logprob(tag: str, word: str) -> float:
    prob = B.get(tag, {}).get(word, UNK)
    return math.log(prob)

def viterbi(tokens: List[str]) -> Tuple[List[str], float]:
    n = len(tokens)
    V = [{}]
    backpointer = [{}]
    
    for tag in TAGS:
        V[0][tag] = math.log(pi[tag]) + emission_logprob(tag, tokens[0])
        backpointer[0][tag] = None
    
    for t in range(1, n):
        V.append({})
        backpointer.append({})
        for tag in TAGS:
            best_prev = None
            max_prob = float('-inf')
            for prev_tag in TAGS:
                prob = V[t-1][prev_tag] + math.log(A[prev_tag][tag]) + emission_logprob(tag, tokens[t])
                if prob > max_prob:
                    max_prob = prob
                    best_prev = prev_tag
            V[t][tag] = max_prob
            backpointer[t][tag] = best_prev
    
    final_best_tag = max(V[n-1], key=lambda tag: V[n-1][tag])
    total_log_prob = V[n-1][final_best_tag]
    
    best_path = [final_best_tag]
    for t in range(n-1, 0, -1):
        best_path.insert(0, backpointer[t][best_path[0]])
    
    return best_path, total_log_prob

sentence1 = ["the", "book", "is", "on", "the", "table"]
sentence2 = ["i", "want", "to", "book", "a", "flight"]

# 运行 Viterbi 解码
tags1, logp1 = viterbi(sentence1)
print("Sentence 1:", list(zip(sentence1, tags1)), "| logP =", round(logp1, 3))

tags2, logp2 = viterbi(sentence2)
print("Sentence 2:", list(zip(sentence2, tags2)), "| logP =", round(logp2, 3))

Sentence 1: [('the', 'DET'), ('book', 'NOUN'), ('is', 'VERB'), ('on', 'PRT'), ('the', 'DET'), ('table', 'NOUN')] | logP = -11.2
Sentence 2: [('i', 'NOUN'), ('want', 'VERB'), ('to', 'PRT'), ('book', 'VERB'), ('a', 'DET'), ('flight', 'NOUN')] | logP = -15.903


**Analysis for Task 4**

* How transition and emission probabilities lead to different tags for "book" in the two sentences.

In the first sentence the word book follows the determiner the. The transition probability from DET to NOUN is high and the emission probability of book from NOUN is higher than from VERB so book is tagged as NOUN. In the second sentence book follows the particle to. The transition probability from PRT to VERB is high and the context want to book suggests a verb so book is tagged as VERB.

* One sentence where the first-order Markov assumption is limiting, and why.

Consider the sentence I saw the man with the telescope. The word with could modify man or saw. The correct interpretation depends on words that are not adjacent. A first-order Markov model only considers the previous tag so it cannot capture this long-range dependency.

### Task 5: Comparing Cosine Similarity and Euclidean Distance (20 Marks)

**Objective:** To empirically demonstrate the difference between angle-based (Cosine) and magnitude-based (Euclidean) similarity measures in a vector space.

**Description:** The choice of similarity metric is crucial. This task highlights how document length affects each metric and why Cosine Similarity is often preferred for text-based topic similarity.

**Your task is to:**

1.  Define three simple documents:
    * `doc_A = "The cat sat on the mat."`
    * `doc_B = "The cat sat on the mat. The dog chased the cat."` (Longer, but on the same topic)
    * `doc_C = "The rocket launched into space."` (Different topic)
2.  Use `sklearn.feature_extraction.text.CountVectorizer` to transform these three documents into count vectors.
3.  Calculate the **Cosine Similarity** between all unique pairs of documents (A-B, A-C, B-C).
4.  Calculate the **Euclidean Distance** between all unique pairs of documents.
5.  Display your results clearly, for instance, in a Pandas DataFrame.
6.  **In a markdown cell, analyze your results:**
    * Explain why the Cosine Similarity between `doc_A` and `doc_B` is high, while their Euclidean Distance is relatively large.
    * Which metric (Cosine Similarity or Euclidean Distance) do your results suggest is better for identifying documents with similar topics, regardless of their length? Justify your answer based on your calculations.

In [7]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
import pandas as pd

doc_A = "The cat sat on the mat."
doc_B = "The cat sat on the mat. The dog chased the cat."
doc_C = "The rocket launched into space."
corpus = [doc_A, doc_B, doc_C]

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(corpus).toarray()

cos_sim = cosine_similarity(X)
euclid_dist = euclidean_distances(X)

results = []
pairs = [("A", "B"), ("A", "C"), ("B", "C")]
for x, y in pairs:
    i = ord(x) - ord("A")
    j = ord(y) - ord("A")
    results.append({
        "Pair": f"{x}-{y}",
        "Cosine Similarity": round(cos_sim[i, j], 4),
        "Euclidean Distance": round(euclid_dist[i, j], 4)
    })

df = pd.DataFrame(results)
print(df.to_string(index=False))

Pair  Cosine Similarity  Euclidean Distance
 A-B             0.9192              2.6458
 A-C             0.3162              3.0000
 B-C             0.3578              4.6904


#### **Analysis for Task 5**

* Explain why the Cosine Similarity between `doc_A` and `doc_B` is high, while their Euclidean Distance is relatively large.

Doc A and doc B talk about the same topic so their word vectors point in a similar direction. Cosine similarity measures direction so it is high. But doc B is longer so its vector has a larger magnitude. Euclidean distance measures absolute difference so it is large.

* Which metric (Cosine Similarity or Euclidean Distance) do your results suggest is better for identifying documents with similar topics, regardless of their length? Justify your answer based on your calculations.

Cosine similarity is better because it ignores document length and focuses only on the relative distribution of words. Euclidean distance is affected by how many times words appear so it is not suitable for comparing topics across documents of different lengths.